In [ ]:
# install
!pip install numpy==1.26.4 -q
!pip install diffusers transformers accelerate anthropic pillow -q
!pip install git+https://github.com/facebookresearch/sam2.git -q
!pip install simple-lama-inpainting easyocr ultralytics -q
!pip install torchao --upgrade -q
!wget -q "https://github.com/google/fonts/raw/main/ofl/bangers/Bangers-Regular.ttf" -O /content/Bangers.ttf

In [ ]:
from google.colab import files

print("Завантаж .py файли")
files.upload()  # llm.py, image_gen.py, consistent_attention.py,
                # bubble_cleaner.py, bubbles.py, pipeline.py

print("Завантаж LoRA weights")
files.upload()  # pytorch_lora_weights.safetensors

print("Завантаж YOLO модель")
files.upload()  # best.pt

In [ ]:
# optional
import os, glob

for fpath in glob.glob("/content/*.py") + glob.glob("/content/*.safetensors") + glob.glob("/content/*.pt"):
    fname = os.path.basename(fpath)
    # remove (1), (2) etc
    import re
    clean = re.sub(r'\s*\(\d+\)', '', fname)
    clean_path = os.path.join("/content", clean)
    if fpath != clean_path:
        os.rename(fpath, clean_path)
        print(f"✓ {fname} → {clean}")

In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = ""  

from image_gen import load_pipeline

pipe = load_pipeline(
    model_id="runwayml/stable-diffusion-v1-5",
    lora_path="/content/pytorch_lora_weights.safetensors",
    device="cuda"
)

In [ ]:
# ui
import ipywidgets as widgets
from IPython.display import display as ipy_display

story_box = widgets.Textarea(
    value="",
    placeholder="Введи свою історію (3-5 речень)...",
    layout=widgets.Layout(width="100%", height="150px")
)
mode_select = widgets.Dropdown(
    options=["adaptive", "standard", "baseline"],
    value="adaptive",
    description="Mode:",
)
decay_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=0.9, step=0.1,
    description="Decay:",
)
seed_input = widgets.IntText(value=42, description="Seed:")
char1_name = widgets.Text(placeholder="ім'я персонажа", description="Char 1:")
char1_desc = widgets.Text(
    placeholder="напр. woman in red coat, dark hair",
    layout=widgets.Layout(width="400px")
)
char2_name = widgets.Text(placeholder="ім'я персонажа", description="Char 2:")
char2_desc = widgets.Text(
    placeholder="напр. translucent ghost, glowing",
    layout=widgets.Layout(width="400px")
)
run_btn  = widgets.Button(description="🎨 Generate Comic", button_style="primary")
comp_btn = widgets.Button(description="🔬 Compare Modes",  button_style="warning")
out      = widgets.Output()

ipy_display(
    widgets.Label("📖 Story:"), story_box,
    widgets.HBox([mode_select, decay_slider, seed_input]),
    widgets.Label("👤 Character overrides (optional):"),
    widgets.HBox([char1_name, char1_desc]),
    widgets.HBox([char2_name, char2_desc]),
    widgets.HBox([run_btn, comp_btn]),
    out
)

def get_overrides():
    overrides = {}
    if char1_name.value.strip() and char1_desc.value.strip():
        overrides[char1_name.value.strip()] = char1_desc.value.strip()
    if char2_name.value.strip() and char2_desc.value.strip():
        overrides[char2_name.value.strip()] = char2_desc.value.strip()
    return overrides or None

def on_generate(b):
    with out:
        out.clear_output()
        import importlib, sys
        for mod in ["llm", "image_gen", "bubble_cleaner", "bubbles", "pipeline"]:
            if mod in sys.modules:
                importlib.reload(sys.modules[mod])
        from pipeline import run_pipeline
        from IPython.display import display
        comic, result, prompts, images = run_pipeline(
            story=story_box.value,
            pipe=pipe,
            mode=mode_select.value,
            decay=decay_slider.value,
            seed=seed_input.value,
            num_inference_steps=50,
            guidance_scale=7.5,
            save_path="/content/comic_output.png",
            character_overrides=get_overrides(),
        )
        display(comic)

def on_compare(b):
    with out:
        out.clear_output()
        import importlib, sys
        for mod in ["llm", "image_gen", "bubble_cleaner", "bubbles", "pipeline"]:
            if mod in sys.modules:
                importlib.reload(sys.modules[mod])
        from pipeline import compare_modes
        from IPython.display import display
        comparison = compare_modes(
            story=story_box.value,
            pipe=pipe,
            save_path="/content/comparison.png",
            character_overrides=get_overrides(),
        )
        display(comparison)

run_btn.on_click(on_generate)
comp_btn.on_click(on_compare)

In [ ]:
from google.colab import files
files.download("/content/comic_output.png")